# 07 — LayoutLM Zone Classifier Training

Fine-tunes LayoutLM-base on annotated financial document pages to
classify document zones (income statement, balance sheet, cash flow,
notes, header, etc.). Uses OCR text + bounding boxes as input.

**Model**: microsoft/layoutlm-base-uncased
**Target accuracy**: 92%+

In [ ]:
# Cell 1: Setup
!pip install -q transformers[torch] datasets accelerate evaluate seqeval tqdm PyYAML mlflow

import json, os
from pathlib import Path
import torch
import yaml
import numpy as np
from transformers import (
    LayoutLMTokenizer,
    LayoutLMForTokenClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from datasets import Dataset, DatasetDict

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

CFG_PATH = Path('../configs/training_config.yaml')
if not CFG_PATH.exists():
    CFG_PATH = Path('/content/drive/MyDrive/numera-ml/configs/training_config.yaml')
cfg = yaml.safe_load(CFG_PATH.read_text())

DATA_DIR = Path(cfg['drive']['data_dir'])
MODELS_DIR = Path(cfg['drive']['models_dir'])
CHECKPOINTS_DIR = Path(cfg['drive']['checkpoints_dir'])
MODELS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

zcfg = cfg['zone_classifier']
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}, Model: {zcfg["model_name"]}')

In [ ]:
# Cell 2: Load annotations and OCR data, build dataset
ZONE_TYPES = ['HEADER', 'INCOME_STATEMENT', 'BALANCE_SHEET', 'CASH_FLOW',
              'EQUITY_CHANGES', 'NOTES', 'TABLE', 'OTHER']
ZONE_TO_ID = {z: i for i, z in enumerate(ZONE_TYPES)}
ID_TO_ZONE = {i: z for z, i in ZONE_TO_ID.items()}

# Load annotations from notebook 06
anno_file = DATA_DIR / 'annotations' / 'zone_annotations.json'
with open(anno_file) as f:
    annotations = json.load(f)

# Load OCR results from notebook 04
ocr_file = DATA_DIR / 'ocr_output' / 'ocr_results.json'
with open(ocr_file) as f:
    ocr_data = json.load(f)

# Build a lookup: image_name → OCR blocks
ocr_lookup = {}
for doc in ocr_data:
    for page in doc['pages']:
        ocr_lookup[page['image']] = page.get('blocks', [])

print(f'Annotations: {len(annotations)} pages, OCR: {len(ocr_lookup)} pages')

In [ ]:
# Cell 3: Prepare LayoutLM input features
tokenizer = LayoutLMTokenizer.from_pretrained(zcfg['model_name'])

def normalize_bbox(bbox, width=1000, height=1000):
    """Normalize bbox to 0-1000 range for LayoutLM."""
    return [
        int(bbox[0] * width / max(bbox[2], 1)),
        int(bbox[1] * height / max(bbox[3], 1)),
        int(min(bbox[2], width)),
        int(min(bbox[3], height)),
    ]

def assign_zone_label(block_bbox, page_zones):
    """Assign zone label to a text block based on its y-position."""
    block_y_center = (block_bbox[1] + block_bbox[3]) / 2
    for zone in page_zones:
        zone_bbox = zone['bbox']
        if zone_bbox[1] <= block_y_center <= zone_bbox[3]:
            return ZONE_TO_ID[zone['type']]
    return ZONE_TO_ID['OTHER']

def prepare_examples(annotations, ocr_lookup, max_length=512):
    """Build tokenized examples for LayoutLM."""
    all_input_ids = []
    all_attention_mask = []
    all_bbox = []
    all_labels = []
    
    for img_name, zones in annotations.items():
        blocks = ocr_lookup.get(img_name, [])
        if not blocks:
            continue
        
        words = []
        word_bboxes = []
        word_labels = []
        
        for block in blocks:
            text = block['text']
            bbox = block.get('bbox', [0, 0, 1, 1])
            label = assign_zone_label(bbox, zones)
            for word in text.split():
                words.append(word)
                word_bboxes.append(normalize_bbox(bbox))
                word_labels.append(label)
        
        if not words:
            continue
        
        encoding = tokenizer(
            words,
            is_split_into_words=True,
            max_length=max_length,
            truncation=True,
            padding='max_length',
            return_tensors='pt',
        )
        
        # Align labels with wordpiece tokens
        word_ids = encoding.word_ids(0)
        aligned_labels = []
        aligned_bboxes = []
        for wid in word_ids:
            if wid is None:
                aligned_labels.append(-100)
                aligned_bboxes.append([0, 0, 0, 0])
            else:
                aligned_labels.append(word_labels[min(wid, len(word_labels) - 1)])
                aligned_bboxes.append(word_bboxes[min(wid, len(word_bboxes) - 1)])
        
        all_input_ids.append(encoding['input_ids'].squeeze())
        all_attention_mask.append(encoding['attention_mask'].squeeze())
        all_bbox.append(torch.tensor(aligned_bboxes))
        all_labels.append(torch.tensor(aligned_labels))
    
    return {
        'input_ids': torch.stack(all_input_ids),
        'attention_mask': torch.stack(all_attention_mask),
        'bbox': torch.stack(all_bbox),
        'labels': torch.stack(all_labels),
    }

features = prepare_examples(annotations, ocr_lookup, max_length=zcfg['max_length'])
print(f'Prepared {features["input_ids"].shape[0]} examples')

In [ ]:
# Cell 4: Train/val split and create dataset
n = features['input_ids'].shape[0]
split = int(n * cfg['data']['train_split'])

indices = torch.randperm(n)
train_idx = indices[:split]
val_idx = indices[split:]

train_ds = Dataset.from_dict({k: v[train_idx].numpy() for k, v in features.items()})
val_ds = Dataset.from_dict({k: v[val_idx].numpy() for k, v in features.items()})

train_ds.set_format('torch')
val_ds.set_format('torch')

print(f'Train: {len(train_ds)}, Val: {len(val_ds)}')

In [ ]:
# Cell 5: Initialize model and training
import evaluate
import mlflow

model = LayoutLMForTokenClassification.from_pretrained(
    zcfg['model_name'],
    num_labels=zcfg['num_labels'],
)
model.to(DEVICE)

metric = evaluate.load('seqeval')

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.argmax(preds, axis=-1)
    true_labels = []
    true_preds = []
    for pred_seq, label_seq in zip(preds, labels):
        t_labels = []
        t_preds = []
        for p, l in zip(pred_seq, label_seq):
            if l != -100:
                t_labels.append(ID_TO_ZONE.get(l, 'OTHER'))
                t_preds.append(ID_TO_ZONE.get(p, 'OTHER'))
        true_labels.append(t_labels)
        true_preds.append(t_preds)
    results = metric.compute(predictions=true_preds, references=true_labels)
    return {
        'precision': results['overall_precision'],
        'recall': results['overall_recall'],
        'f1': results['overall_f1'],
        'accuracy': results['overall_accuracy'],
    }

training_args = TrainingArguments(
    output_dir=str(CHECKPOINTS_DIR / 'layoutlm-zone'),
    num_train_epochs=zcfg['num_epochs'],
    per_device_train_batch_size=zcfg['batch_size'],
    per_device_eval_batch_size=zcfg['batch_size'],
    learning_rate=zcfg['learning_rate'],
    weight_decay=zcfg['weight_decay'],
    warmup_steps=zcfg['warmup_steps'],
    eval_strategy='steps',
    eval_steps=200,
    save_strategy='steps',
    save_steps=zcfg['checkpoint_interval'],
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=50,
    report_to='mlflow',
    fp16=torch.cuda.is_available(),
)

# Configure MLflow
mlflow.set_tracking_uri(cfg['mlflow']['tracking_uri'])
mlflow.set_experiment(zcfg['mlflow_experiment'])

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=zcfg['early_stopping_patience'])],
)

print('Training config ready ✅')
print(f'Epochs: {zcfg["num_epochs"]}, LR: {zcfg["learning_rate"]}, Batch: {zcfg["batch_size"]}')

In [ ]:
# Cell 6: Train!
with mlflow.start_run(run_name='layoutlm-zone-v1'):
    mlflow.log_params(zcfg)
    train_result = trainer.train()
    metrics = trainer.evaluate()
    
    print(f'\nTraining complete!')
    print(f'F1: {metrics["eval_f1"]:.4f}')
    print(f'Accuracy: {metrics["eval_accuracy"]:.4f}')
    
    # Save best model
    model_path = MODELS_DIR / 'layoutlm-zone-classifier'
    trainer.save_model(str(model_path))
    tokenizer.save_pretrained(str(model_path))
    print(f'Model saved to: {model_path}')
    
    # Log to MLflow
    mlflow.log_metrics(metrics)
    mlflow.log_artifact(str(model_path))